# Create a route and display HOSTRADA climate variables

This notebook covers the complete workflow:

1. Select the origin, destination, travel profile, and departure time.
2. Calculate intermediate route points and save them directly as a CSV file.
3. Calculate one or more HOSTRADA climate variables along the route.
4. Display the selected variable as a time series, summary table, and map.

Required files in the same directory:

- `route_leaflet_app_transit_fallback.py`
- `hostradaRoute_all_variables.py`
- the installed or locally available package `hostrada4py`

OSRM is used for driving, cycling, and walking. For rail/public transport,
OpenTripPlanner is tried first; if it is unavailable, the existing
transit fallback is used.


In [ ]:
#pip install -q pandas numpy matplotlib folium ipywidgets

In [1]:
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ipywidgets import Dropdown, SelectMultiple, VBox, Output

from hostrada4py.hostradaRoute import (
    HOSTRADA_VARIABLES,
    available_variables,
    calculate_route_climate,
)

## 1. Create the route and intermediate points

The following values can be selected in the form:

- Origin and destination address
- Departure time
- Driving, cycling, walking, or rail/public transport
- average speed for road profiles
- time interval between intermediate points
- name of the generated CSV file

After clicking **Calculate route**, the CSV file, a preview, and
the route map are generated directly in the notebook.


In [4]:
from hostrada4py.routeLeafletApp import (
    RouteAppDefaults,
    launch_route_app,
)

route_defaults = RouteAppDefaults(
    start_address="Einsteinufer 43-53, 10587 Berlin",
    destination_address="München",
    start_time="",  # leer = aktuelle lokale Time
    average_speed_kmh=50.0,
    interval_minutes=5,
    profile="driving",
    language="de",
    output_file="route_positions.csv",
    otp_url="http://localhost:8080/otp/gtfs/v1",
)

route_app = launch_route_app(route_defaults)

### Use the CSV file from the route form

Run this cell only after the route has been calculated successfully.
The file name entered in the form is automatically used for the
HOSTRADA evaluation.


In [5]:
from pathlib import Path

route_csv = Path(route_app.output_file.value or "route_positions.csv")
output_csv = route_csv.with_name(
    f"{route_csv.stem}_climate.csv"
)

if not route_csv.exists():
    raise FileNotFoundError(
        "The route CSV has not been created yet. "
        "Click 'Calculate route' above and run this cell again. "
        f"Expected file: {route_csv.resolve()}"
    )

print(f"Route CSV: {route_csv.resolve()}")
print(f"Climate output: {output_csv.resolve()}")

Route CSV: /Users/nytschgeusen/GitHub/hostrada4py/route_positions.csv
Climate output: /Users/nytschgeusen/GitHub/hostrada4py/route_positions_climate.csv


## 2. Select and calculate HOSTRADA climate variables

## Available climate variables

In [ ]:
display(available_variables())

## Select climate variables

Select one or more variables. For multiple selection, hold down Ctrl or Cmd
while selecting.

In [7]:
variable_selector = SelectMultiple(
    options=[
        (meta["label"], code)
        for code, meta in HOSTRADA_VARIABLES.items()
    ],
    value=("tas",),
    description="Variables:",
    rows=11,
)

display(variable_selector)

SelectMultiple(description='Variables:', index=(0,), options=(('Air temperature (2 m)', 'tas'), ('Dew point te…

In [8]:
selected_variables = tuple(variable_selector.value)

if not selected_variables:
    raise ValueError("Select at least one climate variable.")

def show_progress(step, total, _row, variable):
    print(
        f"\rCalculation {step}/{total}: {variable}",
        end="",
        flush=True,
    )

climate_data = calculate_route_climate(
    route_csv,
    #variables=selected_variables,
    variables="all", # For a complete export, all variables can alternatively be calculated
    output_csv=output_csv,
    cache_strategy="subset",
    continue_on_error=True,
    progress_callback=show_progress,
)

print(f"\nSaved: {output_csv.resolve()}")
climate_data["timestamp"] = pd.to_datetime(
    climate_data["timestamp"],
    errors="coerce",
)
display(climate_data.head(10))

Calculation 1/528: tasDownload: https://opendata.dwd.de/climate_environment/CDC/grids_germany/hourly/hostrada/air_temperature_mean/tas_1hr_HOSTRADA-v1-0_BE_gn_2026070100-2026073123.nc


KeyboardInterrupt: 

## Select a climate variable for display

The selection contains only variables that were calculated previously.

In [ ]:
plot_selector = Dropdown(
    options=[
        (
            HOSTRADA_VARIABLES[code]["label"],
            code,
        )
        for code in selected_variables
    ],
    value=selected_variables[0],
    description="Chart:",
)

display(plot_selector)

In [ ]:
variable = plot_selector.value
metadata = HOSTRADA_VARIABLES[variable]
value_column = metadata["output_column"]

plot_data = climate_data.dropna(
    subset=["timestamp", value_column]
).sort_values("timestamp")

if plot_data.empty:
    raise ValueError(
        f"No valid values are available for {metadata['label']} vorhanden."
    )

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    plot_data["timestamp"],
    plot_data[value_column],
    marker="o",
)
ax.set_title(f"{metadata['label']} along the route")
ax.set_xlabel("Time")
ax.set_ylabel(
    f"{metadata['short_label']} [{metadata['display_unit']}]"
)
ax.grid(True, alpha=0.3)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## Profile relative to trip start

In [ ]:
route_start = plot_data["timestamp"].iloc[0]
plot_data = plot_data.copy()
plot_data["elapsed_minutes"] = (
    plot_data["timestamp"] - route_start
).dt.total_seconds() / 60

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    plot_data["elapsed_minutes"],
    plot_data[value_column],
    marker="o",
)
ax.set_title(
    f"{metadata['label']} during the trip"
)
ax.set_xlabel("Elapsed travel time [min]")
ax.set_ylabel(
    f"{metadata['short_label']} [{metadata['display_unit']}]"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
summary = pd.DataFrame(
    {
        "Statistic": [
            "Initial value",
            "Final value",
            "Minimum",
            "Maximum",
            "Mean",
            "Change from start → destination",
        ],
        f"Value [{metadata['display_unit']}]": [
            plot_data[value_column].iloc[0],
            plot_data[value_column].iloc[-1],
            plot_data[value_column].min(),
            plot_data[value_column].max(),
            plot_data[value_column].mean(),
            (
                plot_data[value_column].iloc[-1]
                - plot_data[value_column].iloc[0]
            ),
        ],
    }
)
display(summary.round(3))

## Map of the selected climate variable

In [ ]:
map_data = plot_data.dropna(
    subset=["latitude", "longitude", value_column]
)

route_map = folium.Map(
    location=[
        map_data["latitude"].iloc[0],
        map_data["longitude"].iloc[0],
    ],
    zoom_start=8,
    control_scale=True,
)

folium.PolyLine(
    map_data[["latitude", "longitude"]].values.tolist(),
    weight=4,
    opacity=0.8,
    tooltip="Route",
).add_to(route_map)

for _, row in map_data.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        fill=True,
        fill_opacity=0.8,
        tooltip=(
            f"{row['timestamp'].strftime('%Y-%m-%d %H:%M')} · "
            f"{row[value_column]:.2f} {metadata['display_unit']}"
        ),
        popup=(
            f"<b>{metadata['label']}</b><br>"
            f"{row[value_column]:.3f} {metadata['display_unit']}<br>"
            f"<b>Time:</b> {row['timestamp']}"
        ),
    ).add_to(route_map)

route_map.fit_bounds(
    map_data[["latitude", "longitude"]].values.tolist()
)
route_map